In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import pandas as pd


def darken_color(color, factor=0.8):
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)


TARGET_NUM_LOAD_TASKS = '_16'
FILTERS = ['default_st']

RESULT_DIR = "/home/atsushi/ros2-picas/results/"
OUTPUT_DIR = "/home/atsushi/ros2-picas/figure"

TARGET_LABELS = ['C1R1_12_latency', 'C2R3_11_latency', 'C3R2_7_latency', 'C4R3_4_latency']

PLOT_ORDER_MAP = {
    "Default Executors": [
        f'case_study_default_mt4{TARGET_NUM_LOAD_TASKS}',
        f'case_study_default_mt_separate4{TARGET_NUM_LOAD_TASKS}',
    ],
    "Custom Executors": [
        f'case_study_picas_st4{TARGET_NUM_LOAD_TASKS}',
        f'case_study_picas_mt4{TARGET_NUM_LOAD_TASKS}',
        f'case_study_picas_mt_separate4{TARGET_NUM_LOAD_TASKS}',
        f'case_study_cie_4{TARGET_NUM_LOAD_TASKS}',
    ]
}

X_LABEL_MAP = {
    f'case_study_cie_4{TARGET_NUM_LOAD_TASKS}': 'GFP-CIE',
    f'case_study_picas_mt4{TARGET_NUM_LOAD_TASKS}': 'PiCAS-ME (1)',
    f'case_study_picas_mt_separate4{TARGET_NUM_LOAD_TASKS}': 'PiCAS-ME (2)',
    f'case_study_picas_st4{TARGET_NUM_LOAD_TASKS}': 'PiCAS-SE',
    f'case_study_default_mt_separate4{TARGET_NUM_LOAD_TASKS}': 'ME (2)',
    f'case_study_default_mt4{TARGET_NUM_LOAD_TASKS}': 'ME (1)',
}


def read_rt_data(path, key, combined_data):
    with open(path, 'r') as file:
        for line in file:
            parts = line.strip().split()
            if len(parts) != 3 or int(parts[2]) <= 10:
                continue
            label, value = parts[0], int(parts[1])
            if label in combined_data:
                combined_data[label][key].append(value / 1000)  # μs -> ms


# データ読み込みの準備
keys = [d for d in os.listdir(RESULT_DIR)
        if TARGET_NUM_LOAD_TASKS in d and all(f not in d for f in FILTERS)]
combined_data = {label: {key: [] for key in keys} for label in TARGET_LABELS}

for d in keys:
    key = d
    path = os.path.join(RESULT_DIR, d, 'R.txt')
    read_rt_data(path, key, combined_data)

# グラフの描画
for label in TARGET_LABELS:
    default_keys = [k for k in keys if k.startswith("case_study_default_")]
    other_keys = [k for k in keys if not k.startswith("case_study_default_")]

    default_data = [combined_data[label][k] for k in default_keys]
    other_data = [combined_data[label][k] for k in other_keys]

    fig = plt.figure(figsize=(7, 4))
    spec = gridspec.GridSpec(nrows=1, ncols=2, width_ratios=[1, 2])
    axes = [fig.add_subplot(spec[0]), fig.add_subplot(spec[1])]

    for idx, (ax, data_group, group_keys, title, cmap_name) in enumerate(zip(
        axes,
        [default_data, other_data],
        [default_keys, other_keys],
        ["Default Executors", "Custom Executors"],
        ['tab10', 'Set2']
    )):
        ordered_keys = PLOT_ORDER_MAP.get(title, group_keys)
        ordered_keys = [k for k in ordered_keys if k in group_keys]
        ordered_data = [combined_data[label][k] for k in ordered_keys]

        if title == "Default Executors":
            colors = ['#66c2a5', '#fc8d62']  # 任意の2色
        else:
            colors = ["#ffd000", '#e78ac3', '#a6d854', '#8da0cb']  # 任意の4色
        positions = [i + 1 for i in range(len(ordered_data))]

        parts = ax.violinplot(
            dataset=ordered_data,
            positions=positions,
            widths=0.6,
            showmeans=False,
            showextrema=True,
            showmedians=True
        )

        for body, color in zip(parts['bodies'], colors):
            body.set_facecolor(color)
            body.set_edgecolor(color)
            body.set_linewidth(1.5)
            body.set_alpha(0.8)

        for line_key in ['cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes']:
            if line_key in parts:
                lines = parts[line_key]
                if isinstance(lines, list):  # 一般的には list of Line2D
                    for line, color in zip(lines, colors):
                        line.set_color(color)
                        line.set_linewidth(1.5)
                else:
                    # 例：cbars や cmeans は LineCollection の可能性あり
                    lines.set_color(colors)
                    lines.set_linewidth(1.5)

        ax.set_xticks(positions)
        ax.set_xticklabels(
            [X_LABEL_MAP.get(k, k) for k in ordered_keys],
            rotation=30,
            fontsize=14
        )

        # y軸ラベルのフォントサイズを変更
        ax.tick_params(axis='y', labelsize=12)

        if label == 'C4R3_4_latency' and title == "Custom Executors":
            ax.set_ylim(120, 160)
        if label == 'C3R2_7_latency' and title == "Custom Executors":
            ax.set_ylim(30, 80)
        if label == 'C2R3_11_latency' and title == "Custom Executors":
            ax.set_ylim(5, 80)
        if label == 'C1R1_12_latency' and title == "Custom Executors":
            ax.set_ylim(0, 60)

        ax.set_title(title, fontsize=14)
        ax.grid(False)

    if TARGET_NUM_LOAD_TASKS == '_0':
        axes[0].set_ylabel(
            f"{label[:2].replace('C', 'Chain ')}\n\n Response Time [ms]", fontsize=16)

    plt.tight_layout(rect=[0, 0.03, 1, 1.04])
    plt.savefig(f'{OUTPUT_DIR}/{label[:2]}{TARGET_NUM_LOAD_TASKS}.pdf')
    plt.close()

# Context Switch

In [ ]:
import os
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import defaultdict

RESULT_DIR = '/home/atsushi/ros2-picas/results'
TARGET_FILE = 'perf_stat.txt'


def parse_perf_stat(filepath):
    with open(filepath, 'r') as f:
        content = f.read()

    cs_match = re.search(r'([\d,]+)\s+context-switches', content)
    time_match = re.search(r'([\d.]+)\s+seconds time elapsed', content)

    if cs_match and time_match:
        context_switches = int(cs_match.group(1).replace(',', ''))
        time_elapsed = float(time_match.group(1))
        return context_switches / time_elapsed if time_elapsed > 0 else None
    return None


def darken_color(color, factor=0.8):
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)


grouped_data = defaultdict(dict)

for dirname in os.listdir(RESULT_DIR):
    dirpath = os.path.join(RESULT_DIR, dirname)
    filepath = os.path.join(dirpath, TARGET_FILE)

    if not os.path.isfile(filepath):
        continue

    rate = parse_perf_stat(filepath)
    if rate is not None:
        match = re.match(r'^(.+?)_(\d+)$', dirname)
        if match:
            prefix, suffix = match.group(1), int(match.group(2))
            grouped_data[prefix][suffix] = rate

# 凡例と描画順のカスタム設定
LABEL_MAP = {
    'case_study_default_mt4': 'ME (1)',
    'case_study_default_mt_separate4': 'ME (2)',
    'case_study_picas_st4': 'PiCAS-SE',
    'case_study_picas_mt4': 'PiCAS-ME (1)',
    'case_study_picas_mt_separate4': 'PiCAS-ME (2)',
    'case_study_cie_4': 'GFP-CIE',
}


COLOR_MAP = {
    'case_study_default_mt4': '#66c2a5',
    'case_study_default_mt_separate4': '#fc8d62',
    'case_study_picas_st4': '#ffd000',
    'case_study_picas_mt4': '#e78ac3',
    'case_study_picas_mt_separate4': '#a6d854',
    'case_study_cie_4': '#8da0cb',
}

MARKER_MAP = {
    'case_study_default_mt4': 'o',
    'case_study_default_mt_separate4': 's',
    'case_study_picas_st4': '^',
    'case_study_picas_mt4': 'D',
    'case_study_picas_mt_separate4': 'v',
    'case_study_cie_4': 'P',
}

# 凡例表示順
DISPLAY_ORDER = [
    'case_study_default_mt4',
    'case_study_default_mt_separate4',
    'case_study_picas_st4',
    'case_study_picas_mt4',
    'case_study_picas_mt_separate4',
    'case_study_cie_4',
]

plt.figure(figsize=(6, 4))

for prefix in DISPLAY_ORDER:
    if prefix in grouped_data:
        data = grouped_data[prefix]
        x_vals = sorted(data.keys())
        y_vals = [data[x] for x in x_vals]
        plt.plot(
            x_vals,
            y_vals,
            marker=MARKER_MAP.get(prefix, 'o'),
            linestyle='-',
            label=LABEL_MAP.get(prefix, prefix),
            color=darken_color(COLOR_MAP.get(prefix))
        )

plt.xticks([0, 2, 4, 6, 8, 10, 12, 14, 16, 18])

plt.xlabel('Number of Load Tasks', fontsize=14)
plt.ylabel('Number of Context Switches / Sec.', fontsize=14)
plt.grid(False)
plt.legend(fontsize=14)

# X軸のメモリをカンマ区切りにする
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tick_params(axis='both', labelsize=14)

plt.tight_layout()
plt.savefig('/home/atsushi/ros2-picas/figure/context_switches_per_sec.pdf')

# WCRT

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patches as mpatches

RESULT_DIR = '/home/atsushi/ros2-picas/results'

WCRT_S = {
    'C1R1_12_latency': 8,
    'C2R3_11_latency': 19,
    'C3R2_7_latency': 52,
    'C4R3_4_latency': 219,
}


def darken_color(color, factor=0.8):
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)


data = []

# サブディレクトリを探索
for dirname in os.listdir(RESULT_DIR):
    match = re.match(r'^case_study_cie_4_(\d+)$', dirname)
    if match:
        x_value = int(match.group(1))
        r_path = os.path.join(RESULT_DIR, dirname, 'R.txt')
        if os.path.isfile(r_path):
            with open(r_path, 'r') as file:
                for line in file:
                    parts = line.strip().split()
                    if len(parts) >= 2:
                        label = parts[0]
                        try:
                            value = int(parts[1]) / 1000  # μs -> ms
                            data.append((label, x_value, value))
                        except ValueError:
                            continue

# データフレーム化
df = pd.DataFrame(data, columns=['Label', 'X', 'Value'])

color = darken_color('#8da0cb')

# ラベルごとにプロット
unique_labels = df['Label'].unique()
for label in unique_labels:
    subset = df[df['Label'] == label]
    if len(subset['Value']) < 2:
        continue

    plt.figure(figsize=(6, 3))
    sns.boxplot(
        x='X', y='Value', data=subset, showfliers=False, width=0.5,
        boxprops=dict(facecolor='#8da0cb', color=color),
        whiskerprops=dict(color=color),
        capprops=dict(color=color),
        medianprops=dict(color=color))
    # plt.title(f'{label[:2].replace("C", "Chain ")}', fontsize=16)
    plt.xlabel('Number of Stress Tasks')
    plt.ylabel(f'{label[:2].replace("C", "Chain ")}' + '\n\nReponse Time [ms]')
    plt.xticks(rotation=0)
    plt.grid(True)

    # 凡例の追加
    if label in WCRT_S:
        # 分析上の最悪応答時間ライン
        hline = plt.axhline(WCRT_S[label], color='red', linestyle='--', linewidth=1.5,
                            label=f'Analytical WCRT [34]: {WCRT_S[label]}')

        # boxplot 用の凡例項目（色とラベル）
        box_patch = mpatches.Patch(color='#8da0cb', label='GFP-CIE')

        # 凡例の表示（順序：箱 → 解析上限線）
        plt.legend(handles=[hline, box_patch], loc='upper right', bbox_to_anchor=(1.0, 0.90))

    plt.tight_layout()
    plt.savefig(f'/home/atsushi/ros2-picas/figure/{label[:2]}_cie_bound.pdf')